# ML-KEM-768 Benchmark Notebook

Interactive equivalent of `ml_kem_bench.py`. For each selected operation, runs N iterations with random inputs and reports:

- **HW cycles** (median / min / max) — pure accelerator latency from on-chip counter.
- **HW latency** in µs — derived from cycles at 100 MHz.
- **Wall time** seen by Python — includes register writes, cache ops, polling.
- **PYNQ overhead** — (wall − hw), the software control-path cost.
- **Throughput** in ops/s under pulse-then-poll, single thread.

## Op selection

Edit the `op` variable in the **config cell** to choose what to bench:

| `op` value | Behavior |
|---|---|
| `"all"` | Run KeyGen, Encaps, Decaps, Full KEM (default, mirrors `ml_kem_bench.py` no-flag) |
| `"keygen"` | KeyGen only, random `d, z` per iter |
| `"encaps"` | Encaps only, warm `pk` + random `m` |
| `"decaps"` | Decaps only, warm `sk` + warm `ct` (match branch) |
| `"full"` | Full KEM (KG + Enc + Dec) per iter, asserts ss round-trip match |

Equivalent CLI: `sudo python3 ml_kem_bench.py --op <op> -n <n>`.

In [ ]:
import os
import secrets
import statistics
import time

from ml_kem_driver import MLKem768, cycles_to_us


def bench_op(label, fn, n):
    """Run fn() n times. fn must return (..., cycles) or plain cycles."""
    cycles_list = []
    wall_list = []
    for _ in range(n):
        t0 = time.monotonic()
        result = fn()
        wall_list.append(time.monotonic() - t0)
        cyc = result[-1] if isinstance(result, tuple) else result
        cycles_list.append(cyc)

    c_med = statistics.median(cycles_list)
    c_min = min(cycles_list)
    c_max = max(cycles_list)
    w_med = statistics.median(wall_list)
    w_min = min(wall_list)
    w_max = max(wall_list)
    throughput = n / sum(wall_list) if sum(wall_list) > 0 else 0.0

    print(f"  {label}")
    print(f"    HW cycles   : median={int(c_med):6d}  min={c_min:6d}  max={c_max:6d}")
    print(f"    HW latency  : median={cycles_to_us(c_med):6.1f} us  (= {int(c_med)} cyc @ 100 MHz)")
    print(f"    Wall time   : median={w_med*1e6:6.1f} us  min={w_min*1e6:6.1f} us  max={w_max*1e6:6.1f} us")
    print(f"    PYNQ ovhd   : ~{(w_med*1e6 - cycles_to_us(c_med)):6.1f} us  (wall - hw)")
    print(f"    Throughput  : {throughput:6.1f} ops/s  (pulse-then-poll, single-thread)")

    return {
        "label": label,
        "cycles": cycles_list,
        "wall_s": wall_list,
        "cycle_median": c_med,
        "wall_median_s": w_med,
        "throughput_ops_s": throughput,
    }

In [ ]:
# Per-op bench wrappers — each returns the bench_op result dict

def bench_keygen(kem, n):
    print("--- KeyGen (random d, z per iter) ---")
    return bench_op(
        "KeyGen",
        lambda: kem.keygen(secrets.token_bytes(32), secrets.token_bytes(32)),
        n,
    )


def bench_encaps(kem, n):
    print("--- Encaps (warm pk, random m per iter) ---")
    pk_warm, _, _ = kem.keygen(secrets.token_bytes(32), secrets.token_bytes(32))
    return bench_op(
        "Encaps",
        lambda: kem.encaps(pk_warm, secrets.token_bytes(32)),
        n,
    )


def bench_decaps(kem, n):
    print("--- Decaps (warm sk, warm ct - match branch) ---")
    pk_warm, sk_warm, _ = kem.keygen(secrets.token_bytes(32), secrets.token_bytes(32))
    ct_warm, _, _ = kem.encaps(pk_warm, secrets.token_bytes(32))
    return bench_op(
        "Decaps",
        lambda: kem.decaps(sk_warm, ct_warm),
        n,
    )


def bench_full(kem, n):
    print("--- Full KEM round-trip (KG + Encaps + Decaps per iter) ---")

    def full_kem_once():
        d = secrets.token_bytes(32)
        z = secrets.token_bytes(32)
        m = secrets.token_bytes(32)
        pk, sk, c1 = kem.keygen(d, z)
        ct, ss1, c2 = kem.encaps(pk, m)
        ss2, c3 = kem.decaps(sk, ct)
        if ss1 != ss2:
            raise RuntimeError("Round-trip ss mismatch at runtime")
        return (c1 + c2 + c3,)

    return bench_op("Full KEM", full_kem_once, n)


OP_DISPATCH = {
    "keygen": bench_keygen,
    "encaps": bench_encaps,
    "decaps": bench_decaps,
    "full":   bench_full,
}

## Config — edit these and re-run cells below

In [ ]:
bitfile = os.environ.get("ML_KEM_BIT", "./ml_kem_bd.bit")
n  = 100              # iterations per op (50-500 typical)
op = "all"            # one of: "all", "keygen", "encaps", "decaps", "full"

assert op in ("all", "keygen", "encaps", "decaps", "full"), f"unknown op: {op!r}"
print(f"bitfile : {bitfile}")
print(f"n iters : {n}")
print(f"op      : {op}")


In [ ]:
# Open driver (run once per session). If you re-run this cell after a
# previous session, call kem.close() in the cleanup cell below first.
kem = MLKem768(bitfile)
print("Driver loaded.")

In [ ]:
# Run selected op(s). Results are kept in the `results` dict so you can
# inspect raw cycles/wall_s lists afterward (e.g. histogram plotting).
results = {}
if op == "all":
    for name in ["keygen", "encaps", "decaps", "full"]:
        results[name] = OP_DISPATCH[name](kem, n)
        print()
else:
    results[op] = OP_DISPATCH[op](kem, n)

## Optional: inspect raw distribution

After running the bench, `results[op_name]["cycles"]` and `results[op_name]["wall_s"]` hold the per-iteration samples. Useful for histogram plotting, percentile checks, or constant-time verification (decaps Δ=0 across many iters).

In [ ]:
# Example: print per-op summary in CSV-friendly form
for name, r in results.items():
    print(f"{name},"
          f"{int(r['cycle_median'])},"
          f"{cycles_to_us(r['cycle_median']):.2f},"
          f"{r['wall_median_s']*1e6:.2f},"
          f"{r['throughput_ops_s']:.2f}")

In [ ]:
# Cleanup: free CMA buffers. Skip if you intend to keep using `kem`.
kem.close()
print("Driver closed.")

## How to interpret results

- **`HW latency`** comes from the on-chip cycle counter (`REG_CYCLES`) and is the deterministic accelerator compute time. This number should match `tb_ml_kem_top` simulation cycle counts almost exactly.
- **`Wall time`** includes Python/PYNQ overhead: register writes for seeds + control + polling loop + CMA cache flush/invalidate. Typical overhead is ~50-100 µs per op.
- **`PYNQ overhead = wall - hw`** measures the software control-path cost. If this dominates, optimize the driver (e.g. coalesce register writes, use IRQ instead of poll); the RTL itself is already fine.
- **`Throughput`** is ops/s under single-threaded pulse-then-poll. To boost throughput further you would need either DMA-chained IRQ-driven flow or batched register writes.

## Reference cycle budget @ 100 MHz (from simulation)

| Op | HW cycles (sim) | HW latency |
|---|---:|---:|
| KeyGen | ~40k | ~400 µs |
| Encaps | ~57k | ~570 µs |
| Decaps | ~84k | ~840 µs |
| Full KEM | ~181k | ~1.8 ms |

Real on-board numbers from this notebook should match the HW columns within ~1% (deterministic) and add ~50-100 µs of PYNQ overhead per op.